<a href="https://colab.research.google.com/github/ColmTalbot/jax-for-numpy-users/blob/main/jax_for_numpy_users.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Effortless harwdare acceleration and beyond with `JAX`.

## A.k.a.: `JAX` for `numpy users`

This is a notebook prepared as part of a tutorial for the Princeton University research computing workshop series.
Most of the material is adapted directly from the `JAX` documentation.

In [1]:
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("NumPy version:", np.__version__)

JAX version: 0.7.2
NumPy version: 2.0.2


### 1. JAX vs. NumPy Basics

JAX arrays (`jax.numpy`) are very similar to NumPy arrays. You can perform most operations in the same way, but JAX arrays are immutable and designed for automatic differentiation and JIT compilation.

In [2]:
# NumPy example
numpy_array = np.array([1, 2, 3])
print("NumPy array:", numpy_array)

# JAX equivalent
jax_array = jnp.array([1, 2, 3])
print("JAX array:", jax_array)

# Basic operations
print("\nNumPy addition:", numpy_array + 2)
print("JAX addition:", jax_array + 2)

print("\nNumPy dot product:", np.dot(numpy_array, numpy_array))
print("JAX dot product:", jnp.dot(jax_array, jax_array))

NumPy array: [1 2 3]
JAX array: [1 2 3]

NumPy addition: [3 4 5]
JAX addition: [3 4 5]

NumPy dot product: 14
JAX dot product: 14


How does timing compare?
Let's try inverting a reasonably sized matrix.

In [3]:
big_matrix = np.random.rand(3000, 3000)
jax_matrix = jnp.asarray(big_matrix)

We'll use the [`%time`](https://ipython.readthedocs.io/en/9.0.2/interactive/magics.html#magic-time) magic function.

In [4]:
%%time

_ = np.linalg.inv(big_matrix)

CPU times: user 5.01 s, sys: 195 ms, total: 5.21 s
Wall time: 5.68 s


`JAX` functions are asynchronously dispatched, so we need to use `jax.block_until_ready` to make sure the function finishes executing for a [fair timing estimate](https://docs.jax.dev/en/latest/benchmarking.html).

In [5]:
%%time

_ = jax.block_until_ready(jnp.linalg.inv(jax_matrix))

CPU times: user 3.45 s, sys: 154 ms, total: 3.6 s
Wall time: 6.1 s


Isn't `JAX` supposed to be faster than `numpy`?
The first time a `JAX` function is called, it is compiled "just in time" (JIT).
This can add a significant performance overhead for the first call, but subsequent calls should be faster.

In [6]:
%%time

_ = jax.block_until_ready(jnp.linalg.inv(jax_matrix))

CPU times: user 2.51 s, sys: 91.1 ms, total: 2.6 s
Wall time: 2.64 s


I see a modest improvement on average, but since `numpy` is heavily optimized and `colab` doesn't allocate many CPU cores, we shouldn't expect large performance improvements without moving to different hardware.

Try rerunning the above cells using a GPU runtime (`Runtime -> Change runtime type`).
You should see much larger speedups.

### 2. Some more advanced options (`jax.scipy`)

`JAX` also includes an ([incomplete](https://docs.jax.dev/en/latest/jep/18137-numpy-scipy-scope.html)) copy of the `scipy` API.
However, for cases where it is present it can once again be used as a mostly drop-in replacement.
The functionality that is present is generally the subset of `scipy` that is most useful to deep learning applications.

In [7]:
import scipy.signal

CPUs are slow so let's just use a subset of the matrix with `scipy`.

In [8]:
%%time

smaller_matrix = big_matrix[:300, :300]

_ = scipy.signal.convolve2d(smaller_matrix, smaller_matrix)

CPU times: user 51.3 s, sys: 253 ms, total: 51.5 s
Wall time: 51.5 s


This is a good time to choose a GPU runtime if you haven't yet.
You can verify you are using a GPU using the following cell.
You should see `[CudaDevice(id=0)]` or similar if you have access to a GPU.
If you see `[CpuDevice(id=0)]`, you either don't have a GPU available or haven't installed a `CUDA`-compatible version of `JAX`.

In [9]:
jax.devices()

[CpuDevice(id=0)]

In [10]:
%%time

smaller_matrix = jax_matrix[:300, :300]

_ = jax.block_until_ready(jax.scipy.signal.convolve2d(smaller_matrix, smaller_matrix))

CPU times: user 8min 15s, sys: 620 ms, total: 8min 16s
Wall time: 4min 54s


This is a little better, let's see how much of this was compilation time and how fast repeat operations are.

In [11]:
%%time

_ = jax.block_until_ready(jax.scipy.signal.convolve2d(smaller_matrix, smaller_matrix))

CPU times: user 8min 13s, sys: 690 ms, total: 8min 13s
Wall time: 4min 49s


That's more like it.

More complete compatibility between `scipy` and `JAX` is [on the way](https://docs.scipy.org/doc/scipy/dev/api-dev/array_api.html) using the [`Python` array API standard](https://data-apis.org/array-api/latest/index.html).
The implementation is fairly complete, but not yet (as of March 2026) the default setting.

### 3. `numpy` vs `jax.numpy` (a.k.a. [🔪 The Sharp Bits 🔪](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html): Part 1)

While `jax.numpy` provides an almost direct mapping of the `numpy` API, there are a few important differences.

#### 3.1. In-place operations and (im)mutability

In `numpy`, array objects can be pretty much arbitrarily manipulated, e.g.,

In [12]:
numpy_array = np.zeros(4)

print(f"Original array: {numpy_array}")

numpy_array[1] = 4

print(f"Updated array: {numpy_array}")

Original array: [0. 0. 0. 0.]
Updated array: [0. 4. 0. 0.]


In contrast, if we attempt the same thing with a `JAX` array, we will get an error:

In [13]:
jax_array = jnp.zeros(4)

print(f"Original array: {jax_array}")

try:
    jax_array[1] = 4
except TypeError as e:
    print(f"Error: {e}")

print(f"Updated(?) array: {jax_array}")

Original array: [0. 0. 0. 0.]
Error: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html
Updated(?) array: [0. 0. 0. 0.]


`JAX` provides a workaround for this.
Note that this returns a new array and the original is unchanged.


**Note:** The [`array-api-extra`](https://data-apis.org/array-api-extra/generated/array_api_extra.at.html) package, implements the `JAX`-style update functionality for `numpy` arrays.

In [14]:
modified_array = jax_array.at[1].set(4)

print(f"Original array: {jax_array}")
print(f"Updated array: {modified_array}")

Original array: [0. 0. 0. 0.]
Updated array: [0. 4. 0. 0.]


What about in-place operations?

In [15]:
numpy_array_new = numpy_array
numpy_array_new *= 2

jax_array_new = modified_array
jax_array_new *= 2

print(f"Original numpy array: {numpy_array}")
print(f"Updated numpy array: {numpy_array_new}")
print(numpy_array is numpy_array_new)

print(f"Original jax array: {jax_array}")
print(f"Updated jax array: {jax_array_new}")
print(jax_array is jax_array_new)

Original numpy array: [0. 8. 0. 0.]
Updated numpy array: [0. 8. 0. 0.]
True
Original jax array: [0. 0. 0. 0.]
Updated jax array: [0. 8. 0. 0.]
False


Here we can see that using the in-place multiplication operator with `JAX` has returned a new object rather than modifying the existing object in place.

#### 3.2 Type promotion

In `numpy` it is common to pass non-`numpy` objects to numpy functions, for example

In [16]:
np.sum([1, 2, 3])

np.int64(6)

In here there is an implicit type-promotion.
`numpy.sum` acts on `numpy` array objects and so the provided list is first silently converted to an array, and then the operation is applied.

In `JAX`, this can lead to serious performance problems and so, the inputs must be explicitly cast to the correct type.

In [17]:
try:
    jnp.sum([1, 2, 3])
except TypeError as e:
    print(f"Error: {e}")

jnp.sum(jnp.asarray([1, 2, 3]))

Error: sum requires ndarray or scalar arguments, got <class 'list'> at position 0.


Array(6, dtype=int32)

Note that `JAX` will accept `numpy` arrays as arguments, typically within `JIT`-compiled functions this will not lead to major performance issues, but it can lead to issues.

In [18]:
jnp.sum(np.asarray([1, 2, 3]))

Array(6, dtype=int32)

#### 3.3 Default precision

By default, `numpy` will assume all float arrays should use 64-bit precision.

Since `JAX` was designed to work with deep learning workflows where speed is more important than precision, the default precision for floats is 32-bit.

For many scientific applications, 64-bit precision is required to ensure accuracy.
You can enable 64-bit precision in `JAX` with

In [19]:
jax.config.update("jax_enable_x64", True)

jnp.sum(np.asarray([1.0, 2.0, 3.0]))

Array(6., dtype=float64)

This can also be specified as an environment variable with `JAX_ENABLE_X64=True`.

If you see an immediate 2x increase in speed when switching to `JAX`, it is likely due to the difference in precision.

#### 3.4 Out of bounds indexing

In `numpy`, querying an index outside the array will raise an error

In [20]:
try:
    np.arange(5)[6]
except IndexError as e:
    print(f"Error: {e}")

Error: index 6 is out of bounds for axis 0 with size 5


In `JAX`, this will happily return the last element of the array which can lead to very confusing bugs!

In [21]:
jnp.arange(5)[6]

Array(4, dtype=int64)

#### 3.5. Random number generation

`numpy` random number generation relies on defining a random number generator (RNG) state, which is modified in place each time a random method is called.

In [22]:
rng = np.random.default_rng(42)

rng.uniform()

0.7739560485559633

In [23]:
rng = np.random.default_rng(42)

print(f"Initial random state: {rng.bit_generator.state}")

print(f"First random number: {rng.uniform()}")

print(f"Second random state: {rng.bit_generator.state}")

print(f"Second random number: {rng.uniform()}")

print(f"Final random state: {rng.bit_generator.state}")

Initial random state: {'bit_generator': 'PCG64', 'state': {'state': 274674114334540486603088602300644985544, 'inc': 332724090758049132448979897138935081983}, 'has_uint32': 0, 'uinteger': 0}
First random number: 0.7739560485559633
Second random state: {'bit_generator': 'PCG64', 'state': {'state': 95376830783351682500486580248632280039, 'inc': 332724090758049132448979897138935081983}, 'has_uint32': 0, 'uinteger': 0}
Second random number: 0.4388784397520523
Final random state: {'bit_generator': 'PCG64', 'state': {'state': 180456145198944327624639367796045148994, 'inc': 332724090758049132448979897138935081983}, 'has_uint32': 0, 'uinteger': 0}


This is fully reproducible, however, it relies on a stateful generator

Since `JAX` makes a strong assumption that functions and methods should not change the objects they act on, they need a different approach.

The core of random number generation in `JAX` relies on passing an explicit "random key".

**Note:** This is our first use of `JAX` functionality outside `jax.numpy`.

In [24]:
rng_key = jax.random.key(42)

print(f"First random state: {rng_key}")

print(f"First random number: {jax.random.normal(rng_key)}")

print(f"Second random state: {rng_key}")

print(f"Second random number: {jax.random.normal(rng_key)}")

print(f"Final random state: {rng_key}")

First random state: Array((), dtype=key<fry>) overlaying:
[ 0 42]
First random number: -0.18471174528191162
Second random state: Array((), dtype=key<fry>) overlaying:
[ 0 42]
Second random number: -0.18471174528191162
Final random state: Array((), dtype=key<fry>) overlaying:
[ 0 42]


Both of these random numbers are the same!
_The `JAX` random functions do not modify the RNG key._

Instead, we must explicitly "split" the key every time for example as follows.

In [25]:
rng_key, this_key = jax.random.split(rng_key)

print(f"Split random state: {rng_key}, {this_key}")

print(f"New random number: {jax.random.normal(this_key)}")

Split random state: Array((), dtype=key<fry>) overlaying:
[1832780943  270669613], Array((), dtype=key<fry>) overlaying:
[  64467757 2916123636]
New random number: 0.6498920118351406


You can also generate many new keys simultaneously

In [26]:
rng_key, *other_keys = jax.random.split(rng_key, 5)

print(f"Carried random state: {rng_key}")

print(f"{len(other_keys)} generated random states: {other_keys}")

Carried random state: Array((), dtype=key<fry>) overlaying:
[1012194634 3152801799]
4 generated random states: [Array((), dtype=key<fry>) overlaying:
[1705926158  899080142], Array((), dtype=key<fry>) overlaying:
[4095997477  317277840], Array((), dtype=key<fry>) overlaying:
[1820970612 3729538270], Array((), dtype=key<fry>) overlaying:
[3985668294 1940518238]]


**Note:** Unifying the two random number generation methods described above is [not currently in scope](https://github.com/data-apis/array-api/issues/431) for the Python array API project.

### 4. Just in time (JIT) compilation

Beside hardware transparency, JIT-compilation is probably the most significant source of performance gains compared with `numpy`.

The basic syntax is using the `jax.jit` [decorator](https://peps.python.org/pep-0318/).

In [27]:
def my_test_function(
    input_array_1: jax.Array, input_array_2
) -> jax.Array:

    intermediate_array = input_array_1 + input_array_2
    output_array = intermediate_array @ input_array_1

    return jnp.sum(output_array)

In [28]:
%%time

_ = jax.block_until_ready(my_test_function(
    jnp.linspace(4, 10, 10000),
    jnp.arange(11, 12, 10000)
))

CPU times: user 174 ms, sys: 5.95 ms, total: 180 ms
Wall time: 178 ms


Let's make a copy of the same function but with the `jax.jit` decorator.

**Note:** this is equivalent to

```python
my_jitted_function = jax.jit(my_test_function)
```

In [29]:
@jax.jit
def my_jitted_function(
    input_array_1: jax.Array, input_array_2
) -> jax.Array:

    intermediate_array = input_array_1 + input_array_2
    output_array = intermediate_array @ input_array_1

    return jnp.sum(output_array)

We'll run the JIT-ted function twice.
The first time will be slow as the function will be compiled.
The second evaluation should be much faster.

In [30]:
%%time

_ = jax.block_until_ready(my_jitted_function(
    jnp.linspace(4, 10, 10000),
    jnp.arange(11, 12, 10000)
))

CPU times: user 67.1 ms, sys: 4.95 ms, total: 72 ms
Wall time: 57.6 ms


In [31]:
%%time

_ = jax.block_until_ready(my_jitted_function(
    jnp.linspace(4, 10, 10000),
    jnp.arange(11, 12, 10000)
))

CPU times: user 2.24 ms, sys: 1 µs, total: 2.24 ms
Wall time: 1.88 ms


Let's try evaluating again with different inputs

In [32]:
%%time

_ = jax.block_until_ready(my_jitted_function(
    jnp.linspace(4, 10, 1000),
    jnp.arange(11, 12, 1000)
))

CPU times: user 89.8 ms, sys: 2.95 ms, total: 92.8 ms
Wall time: 91.8 ms


It's slow again!

Let's take a closer look at the compilation.
There are two useful diagnostic tools for understanding when compilation is triggered and why.

[`jax.log_compiles`](https://docs.jax.dev/en/latest/_autosummary/jax.log_compiles.html) will print lots of information about the compilation.
For now, we won't worry about all the details being printed.
The important point is that with this context active, if recompilation is triggered we will see some printed messages.
[`jax.explain_cache_misses`](https://docs.jax.dev/en/latest/config_options.html#jax_explain_cache_misses) will also trigger logged messages when recompilation is needed along with some information about the input arguments.

In [33]:
print("Trigger a recompilation with explain_cache_misses")

with jax.explain_cache_misses():
    _ = jax.block_until_ready(my_jitted_function(
        jnp.linspace(4, 10, 4000),
        jnp.arange(11, 12, 4000)
    ))

print("\n" * 4)

print("Trigger a recompilation with log_compiles")

with jax.log_compiles():
    _ = jax.block_until_ready(my_jitted_function(
        jnp.linspace(4, 10, 400, dtype=int),
        jnp.arange(11, 12, 4000, dtype=int)
    ))


Trigger a recompilation with explain_cache_misses


  for my_jitted_function defined at /tmp/ipykernel_250/3680736903.py:1
  all previously seen cache keys are different. Several previous keys are closest:
  * key with different input types:
      types now: input_array_1: f64[4000], input_array_2: i64[1]
        * at input_array_1, now f64[4000] and before f64[10000]
  * key with different input types:
      types now: input_array_1: f64[4000], input_array_2: i64[1]
        * at input_array_1, now f64[4000] and before f64[1000]
  for my_jitted_function defined at /tmp/ipykernel_250/3680736903.py:1
  all previously seen cache keys are different. Several previous keys are closest:
  * key with different input types:
      types now: input_array_1: f64[4000], input_array_2: i64[1]
        * at input_array_1, now f64[4000] and before f64[10000]
  * key with different input types:
      types now: input_array_1: f64[4000], input_array_2: i64[1]
        * at input_array_1, now f64[4000] and before f64[1000]







Trigger a recompilation with log_compiles


We can see from the first message that the function is being recompiled because the shape of the inputs has changed.
The `XLA` compilation is specific to the shape of the inputs.
Changing the type of the inputs also requires a recompilation.

While this can lead to extremely dramatic speed ups for repeated evaluations, it means that if the input shapes are different every time a function is evaluated, it may not be a good target for compilation.

As a verification, we can check that compilation is not retriggered when evaluating with arrays of the same shape with different values.

In [34]:
with jax.explain_cache_misses():
    _ = jax.block_until_ready(my_jitted_function(
        jnp.linspace(5, 10, 4000),
        jnp.arange(11, 2002, 4000)
    ))


#### 4.1 Control flow and JIT

`XLA` compilation relies on knowing the exact sequence of operations that will be applied to the inputs and so control flow operations that cause branching in the evaluation tree pose significant challenges.

The following function will raise an error as the operations depend on the value of one of the input arguments.

In [35]:
@jax.jit
def condition_evaluation(input_array, value):
    if value < 5:
        return input_array
    else:
        return input_array * 2


try:
    condition_evaluation(jnp.ones(4), 1)
except jax._src.core.TracerBoolConversionError as e:
    print(f"Raised error: {e}")

Raised error: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function condition_evaluation at /tmp/ipykernel_250/436955350.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument value.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


We see a nice error message with a link to the `JAX` documentation.

There are a few references to "tracing" in the message.
During the compilation, `JAX` traces the operations applied to each of the arguments and so within JIT compiled functions all arrays are replaced with trace objects that include the type and shape of the array.

We can avoid this by specifying that we want `JAX` to recompile the function for every value of the second argument using the `static_argnames` argument

In [36]:
def condition_evaluation(input_array, value):
    if value < 5:
        return input_array
    else:
        return input_array * 2


with jax.explain_cache_misses():
    jit_compiled = jax.jit(condition_evaluation, static_argnames="value")
    print(jit_compiled(jnp.ones(4), 1))
    print(jit_compiled(jnp.ones(4), 6))

  never seen function:
    condition_evaluation id=137616409982464 defined at /tmp/ipykernel_250/309444276.py:1
  never seen function:
    condition_evaluation id=137616409982464 defined at /tmp/ipykernel_250/309444276.py:1
  for condition_evaluation defined at /tmp/ipykernel_250/309444276.py:1
  all previously seen cache keys are different. Closest previous key:
  * key with different value of static args:
      now 6 and before 1
  for condition_evaluation defined at /tmp/ipykernel_250/309444276.py:1
  all previously seen cache keys are different. Closest previous key:
  * key with different value of static args:
      now 6 and before 1


[1. 1. 1. 1.]
[2. 2. 2. 2.]


This successfully compiles, but at the cost of an increased number of compilations.

A related failure mode is returning a slice of an array based on the value of an argument.

In [37]:
@jax.jit
def condition_evaluation(input_array, n_keep):
    return input_array[:n_keep]


try:
    condition_evaluation(jnp.ones(4), 1)
except IndexError as e:
    print(f"Raised error: {e}")

Raised error: Array slice indices must have static start/stop/step to be used with NumPy indexing syntax. Found slice(None, JitTracer<~int64[]>, None). To index a statically sized array at a dynamic position, try lax.dynamic_slice/dynamic_update_slice (JAX does not support dynamically sized arrays within JIT compiled functions).


Again, we get a useful error.
I'm not going to get into dynamic slices here, but they can enable some flexibility in this case.

Since the shape of the input arrays is fixed at compile time, we can define arbitrary control flow based on those shapes, e.g.,

In [38]:
@jax.jit
def condition_evaluation(input_array):
    if input_array.shape[0] < 5:
        return input_array
    else:
        return input_array * 2


print(condition_evaluation(jnp.ones(4)))
print(condition_evaluation(jnp.ones(10)))

[1. 1. 1. 1.]
[2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]


#### 4.2 Extra code

JIT compilation will strip out any portion of the function that is not traced.
The most obvious example is print statements

In [39]:
@jax.jit
def function_with_print(input_array):
    print(f"The input array is {input_array}")
    return input_array.sum()


print(function_with_print(jnp.ones(4)))
print(function_with_print(jnp.ones(4)))
print(function_with_print(jnp.ones(4)))

The input array is JitTracer<float64[4]>
4.0
4.0
4.0


We see two interesting things here:
- The print statement only appears once.
  This is because the print statement has been removed by the compiled.
- The value of the input array has been replaced with a `JitTracer`.
  This is because the print occurs as the function is being traced by JIT and so the input argument has been replaced with a tracer.

This can make debugging what is happening inside JIT compiled functions difficult and so the `JAX` developers provided a print function that will work within compiled functions.

In [40]:
@jax.jit
def function_with_print(input_array):
    jax.debug.print("The input array is {}", input_array)
    return input_array.sum()


print(function_with_print(jnp.ones(4)))
print(function_with_print(jnp.ones(4)))
print(function_with_print(jnp.ones(4)))

The input array is [1. 1. 1. 1.]
4.0
The input array is [1. 1. 1. 1.]
4.0
The input array is [1. 1. 1. 1.]
4.0


The full details are provided in `JAX` [documentation](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html#jax.debug.print), but the most important points are that we cannot use f-strings within these statements and that there can be performance penalties to including these statements.

Just to demonstrate that the removal of unused code is generic let's look at another case

In [41]:
import time

@jax.jit
def function_with_print(input_array):
    time.sleep(5)
    return input_array.sum()


print("The first evaluation will sleep for five seconds")
print(function_with_print(jnp.ones(4)))
print("The rest will return immediately")
print(function_with_print(jnp.ones(4)))
print(function_with_print(jnp.ones(4)))

The first evaluation will sleep for five seconds
4.0
The rest will return immediately
4.0
4.0


### 5. Automatic Differentiation (autodiff)

Automatic differentiation provides an exact derivative of any compatible function in a single evaluation of the function.
By comparison, numerically computing the gradient of a function with N-dimensional input requires N evaluations.

#### 5.1 Scalar functions

We'll start with a simple scalar function with scalar input.
$$f(x) = x^2 + 2x + 1$$
with derivatives
$$f'(x) = 2x + 2$$
$$f''(x) = 2$$
$$f'''(x) = 0$$

In [42]:
def scalar_function(x):
    return x**2 + 2*x + 1

# Compute the gradient of the function
grad_scalar_function = jax.grad(scalar_function)

# Evaluate the function and its gradient at a point
x_val = 3.0
print(f"Function value at x={x_val}: {scalar_function(x_val)}")
print(f"Gradient value at x={x_val}: {grad_scalar_function(x_val)}")

Function value at x=3.0: 16.0
Gradient value at x=3.0: 8.0


We can evaluate higher order derivatives by just stacking `grad` calls.

In [43]:
second_derivative = jax.grad(grad_scalar_function)
third_derivative = jax.grad(second_derivative)
print(f"Second derivative at x={x_val}: {second_derivative(x_val)}")
print(f"Third derivative at x={x_val}: {third_derivative(x_val)}")

Second derivative at x=3.0: 2.0
Third derivative at x=3.0: 0.0


We can additionally calculate derivatives of multivariate functions and even specify which inputs we want to calculate the derivative with respect to using the `argnums` variable.

In [44]:
def multivariable_function(x, y):
    return x**2 + y**3


grad_x = jax.grad(multivariable_function, argnums=0)
print(f"\nGradient w.r.t. x at (3, 2): {grad_x(3.0, 2.0)}")

grad_y = jax.grad(multivariable_function, argnums=1)
print(f"Gradient w.r.t. y at (3, 2): {grad_y(3.0, 2.0)}")

grad_xy = jax.grad(multivariable_function, argnums=(0, 1))
print(f"Gradient w.r.t. x and y at (3, 2): {grad_xy(3.0, 2.0)}")


Gradient w.r.t. x at (3, 2): 6.0
Gradient w.r.t. y at (3, 2): 12.0
Gradient w.r.t. x and y at (3, 2): (Array(6., dtype=float64, weak_type=True), Array(12., dtype=float64, weak_type=True))


Our third simple application is a scalar function of a vector input.
$$f(x) = \sum_i^{N} x^i$$
with gradient
$$f'(x) = (i x^{i - 1}; 0 \leq i \leq N)$$

In [45]:
def vector_input_function(x):
    return jnp.sum(x**np.arange(x.shape[0]))


point = 2 * jnp.ones(4)

print(f"Gradient at {point}: {jax.grad(vector_input_function)(point)}")

Gradient at [2. 2. 2. 2.]: [ 0.  1.  4. 12.]


In this case, we can't take higher-order derivatives using `jax.grad` since the gradient is no longer a scalar-output function.

In [46]:
try:
    jax.grad(jax.grad(vector_input_function))(point)
except TypeError as e:
    print(f"Raised error: {e}")

Raised error: Gradient only defined for scalar-output functions. Output had shape: (4,).


#### 5.2 Jacobians

The gradient of vector-valued functions
$$
y_j = f(x_i)
$$
of vector-valued inputs is the [Jacobian matrix](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant)
$$
J = \frac{\partial y_j}{\partial x_i}.
$$

There are two widely-used autodiff methods for computing Jacobians known as "forward" (jacobian-vector product, `jax.jacfwd`) and "reverse" (vector-jacobian product, `jax.jacrev`) mode.
For more details on the implementations in `JAX` see the [documentation](https://docs.jax.dev/en/latest/jacobian-vector-products.html).

In [47]:
def vector_function(x):
    return x**np.arange(x.shape[0]) + x[::-1]**(2 + np.arange(x.shape[0]))


point = 5 + 2 * jnp.arange(1, 5, dtype=float)

print(f"JVP at {point}: {jax.jacfwd(vector_function)(point)}")
print(f"VJP at {point}: {jax.jacrev(vector_function)(point)}")

JVP at [ 7.  9. 11. 13.]: [[0.0000e+00 0.0000e+00 0.0000e+00 2.6000e+01]
 [0.0000e+00 1.0000e+00 3.6300e+02 0.0000e+00]
 [0.0000e+00 2.9160e+03 2.2000e+01 0.0000e+00]
 [1.2005e+04 0.0000e+00 0.0000e+00 5.0700e+02]]
VJP at [ 7.  9. 11. 13.]: [[0.0000e+00 0.0000e+00 0.0000e+00 2.6000e+01]
 [0.0000e+00 1.0000e+00 3.6300e+02 0.0000e+00]
 [0.0000e+00 2.9160e+03 2.2000e+01 0.0000e+00]
 [1.2005e+04 0.0000e+00 0.0000e+00 5.0700e+02]]


These two methods return the same result, the difference is purely algorithmic.
In general, if the input (output) vector is lower dimensional than the output (input) vector use forward (reverse) mode*.


* This is why reverse mode autodiff is the default in most machine learning frameworks, `jax.grad` uses reverse mode.

#### 5.3 More complex functions

What if we want to differentiate over more complex inputs?
For example, what about a dictionary input of scalars that returns a dictionary of arrays?

In [48]:
def complex_function(input_dict):
    output_dict = {}

    for key_1, value_1 in input_dict.items():
        for key_2, value_2 in input_dict.items():
            # control flow is allowed based on dictionary keys
            if "a" in key_1 and "b" in key_2:
                output_dict[(key_1, key_2)] = value_1 * value_2
            else:
                output_dict[(key_1, key_2)] = (
                    value_1**np.arange(5)
                    + value_2**np.arange(5)[::-1]
                )

    return output_dict


complex_point = dict(
    a_1=2.0,
    a_2=3.0,
    b_1=4.0,
    b_2=5.0,
)
jax.jacfwd(complex_function)(complex_point)

{('a_1', 'a_1'): {'a_1': Array([32., 13.,  8., 13., 32.], dtype=float64),
  'a_2': Array([0., 0., 0., 0., 0.], dtype=float64),
  'b_1': Array([0., 0., 0., 0., 0.], dtype=float64),
  'b_2': Array([0., 0., 0., 0., 0.], dtype=float64)},
 ('a_1', 'a_2'): {'a_1': Array([ 0.,  1.,  4., 12., 32.], dtype=float64),
  'a_2': Array([108.,  27.,   6.,   1.,   0.], dtype=float64),
  'b_1': Array([0., 0., 0., 0., 0.], dtype=float64),
  'b_2': Array([0., 0., 0., 0., 0.], dtype=float64)},
 ('a_1', 'b_1'): {'a_1': Array(4., dtype=float64, weak_type=True),
  'a_2': Array(0., dtype=float64, weak_type=True),
  'b_1': Array(2., dtype=float64, weak_type=True),
  'b_2': Array(0., dtype=float64, weak_type=True)},
 ('a_1', 'b_2'): {'a_1': Array(5., dtype=float64, weak_type=True),
  'a_2': Array(0., dtype=float64, weak_type=True),
  'b_1': Array(0., dtype=float64, weak_type=True),
  'b_2': Array(2., dtype=float64, weak_type=True)},
 ('a_2', 'a_1'): {'a_1': Array([32., 12.,  4.,  1.,  0.], dtype=float64),
  'a_2

This works because `JAX` transformations (e.g., `jax.jit`, `jax.jacfwd`, `jax.grad`) are defined on [pytree](https://docs.jax.dev/en/latest/pytrees.html) objects.
`JAX` contains pytree implementations for many built-in `Python` types, which can be arbitrarily nested.
Users can even define their own pytrees for classes.

The flexibility of autodiff in `JAX` is pretty remarkable.
If you want to compute a derivative over some input and are worried it won't work, try it.
It might just work.

### 6. Vectorization (`jax.vmap`)

`jax.vmap` is used to automatically vectorize a function that was originally written for a single input. This is extremely useful for batching operations without explicitly writing loops or changing your function's signature.

Unlike the `numpy.vectorize` funtion which is just a thin wrapper for a `for` loop, `jax.vmap` (and `jax.numpy.vectorize`) will actually parallelize the calculation as much as possible.

#### 6.1 Basic usage

The most basic usage is to automate an operation over the leading axis of the arguments of a function.

In [49]:
def sum_of_squares_single(vector):
    return jnp.sum(vector**2)


vmapped_sum_of_squares = jax.vmap(sum_of_squares_single)

batch_of_vectors = jnp.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("Input batch:\n", batch_of_vectors)
print("\nOutput of vmapped function:\n", vmapped_sum_of_squares(batch_of_vectors))

manual_results = jnp.array([sum_of_squares_single(vec) for vec in batch_of_vectors])
print("\nManual loop results:\n", manual_results)

Input batch:
 [[1. 2. 3.]
 [4. 5. 6.]]

Output of vmapped function:
 [14. 77.]

Manual loop results:
 [14. 77.]


#### 6.2 Map across pytrees

Since `jax.vmap` is another transformation, it can be applied across any pytree.
We can demonstrate this looking at our function over dictionaries defined above and passing dictionaries containing arrays.

In [50]:
dict_of_vectors = dict(
    a_1=jnp.linspace(1, 2, 15),
    a_2=jnp.linspace(3, 4, 15),
    b_1=jnp.linspace(5, 6, 15),
    b_2=jnp.linspace(1, 6, 15),
)

try:
    complex_function(dict_of_vectors)
except TypeError as e:
    print(f"Raised error: {e}")

Raised error: pow got incompatible shapes for broadcasting: (15,), (5,).


This raises an error because of attempting to raise an array to the power of another array which cannot be broadcast together by default.
We could rewrite this by manually implementing broadcasting rules for the problematic operation.
Or we can just `jax.vmap`.

In [51]:
result = jax.vmap(complex_function)(dict_of_vectors)

print("Output contains:")
for key in result:
    print(f"\t{key} with shape {result[key].shape}")

Output contains:
	('a_1', 'a_1') with shape (15, 5)
	('a_1', 'a_2') with shape (15, 5)
	('a_1', 'b_1') with shape (15,)
	('a_1', 'b_2') with shape (15,)
	('a_2', 'a_1') with shape (15, 5)
	('a_2', 'a_2') with shape (15, 5)
	('a_2', 'b_1') with shape (15,)
	('a_2', 'b_2') with shape (15,)
	('b_1', 'a_1') with shape (15, 5)
	('b_1', 'a_2') with shape (15, 5)
	('b_1', 'b_1') with shape (15, 5)
	('b_1', 'b_2') with shape (15, 5)
	('b_2', 'a_1') with shape (15, 5)
	('b_2', 'a_2') with shape (15, 5)
	('b_2', 'b_1') with shape (15, 5)
	('b_2', 'b_2') with shape (15, 5)


#### 6.3 Combining transformations

All `JAX` transformations are composable, i.e., you can write `jax.jit(jax.jacfwd(jax.vmap(jax.grad(jax...(my_function)))))` and `JAX` will happily resolve these operations.

Figuring out the optimal ordering of these can be non-trivial.
However, as a general rule, if a composed function will be reused, you can wrap it in a final `jax.jit` layer.

In [52]:
dict_of_big_vectors = dict(
    a_1=jnp.linspace(1, 2, 1500),
    a_2=jnp.linspace(3, 4, 1500),
    b_1=jnp.linspace(5, 6, 1500),
    b_2=jnp.linspace(1, 6, 1500),
)

jit_then_vmap = jax.vmap(jax.jit(complex_function))
_ = jit_then_vmap(dict_of_big_vectors)
vmap_then_jit = jax.jit(jax.vmap(complex_function))
_ = vmap_then_jit(dict_of_big_vectors)

In [53]:
%%time

_ = jax.block_until_ready(jit_then_vmap(dict_of_big_vectors))

CPU times: user 5.31 ms, sys: 0 ns, total: 5.31 ms
Wall time: 4.39 ms


In [54]:
%%time

_ = jax.block_until_ready(vmap_then_jit(dict_of_big_vectors))

CPU times: user 1.71 ms, sys: 993 µs, total: 2.71 ms
Wall time: 2.5 ms


`jax.vmap` can also be used as a decorator

In [55]:
@jax.vmap
def my_scalar_function(x):
    return x**2


my_scalar_function(jnp.linspace(0, 1, 10))

Array([0.        , 0.01234568, 0.04938272, 0.11111111, 0.19753086,
       0.30864198, 0.44444444, 0.60493827, 0.79012346, 1.        ],      dtype=float64)

#### 6.4 Additional arguments

The behaviour of `jax.vmap` can be customised to specify over which input and output arguments of each argument should be mapped over.
The semantics of this are described in [the docs](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html).

In [56]:
def function_with_unmatched_arguments(x, y):
    return (x * y).sum()


x = jnp.ones((3, 4))
y = jnp.ones((4, 3))

try:
    jax.vmap(function_with_unmatched_arguments)(x, y)
except ValueError as e:
    print(f"Raised error: {e}")

Raised error: vmap got inconsistent sizes for array axes to be mapped:
  * one axis had size 3: axis 0 of argument x of type float64[3,4];
  * one axis had size 4: axis 0 of argument y of type float64[4,3]


This raised an error because we really want to map over the second axis of `x` and the first axis of `y`.
We need to customize our `vmap` call.

In [57]:
jax.vmap(function_with_unmatched_arguments, in_axes=(1, 0))(x, y)

Array([3., 3., 3., 3.], dtype=float64)

**Note:** `jax.vmap` will attempt to perform the entire operation at once and so can have large memory usage for some functions.
If you find `jax.vmap` leads to memory issues, you can still map over arbitrary pytrees in serial using `jax.tree.map`.

### 7. A Simple Neural Network Example

Combining some of the operations above, we can build and train a simple neural network. Here's a minimal example of training a linear model (a single-layer neural network) using gradient descent.

The model here is
$$
f(x) = W x + b
$$
where x and b are vectors and W is a matrix.

First, we need to define some helper functions to evaluate our model, draw an initial sample, and compute the loss.

In [58]:
def init_params(key):
    """Draw a random set of our parameters to start our training loop"""
    w_key, b_key = jax.random.split(key)
    W = jax.random.normal(w_key, (2, 3))
    b = jax.random.normal(b_key, (2,))
    return dict(W=W, b=b)


@partial(jax.vmap, in_axes=(None, 0))
def linear_model(params, x):
    """
    Evaluate our linear model for a single parameter vector (x)

    We wrap this in vmap to enable automatic batching of the model.
    Note that the `@partial(jax.vmap, ...)` or `@partial(jax.jit, ...)`
    is a common pattern to pass additional arguments while using the
    decorator functionality.
    """
    return params["W"] @ x + params["b"]


@jax.jit
def loss_fn(params, x, y):
    predictions = linear_model(params, x)
    return jnp.mean((predictions - y)**2)

Next we can set up our true values and simulate some data.

We also use the `jax.value_and_grad` transformation, to get a function that will evaluate in one go the value of the loss and the reverse-mode gradient.

In [59]:
key = jax.random.key(0)
jey, this_key = jax.random.split(key)
X = jax.random.normal(key, (100, 3, 2))
W = jnp.array([[3.0, 4.0, 5.0], [1.0, 7.0, 5.3]])
b = jnp.array([5.2, 1.9])
y = linear_model(dict(W=W, b=b), X)

key, this_key = jax.random.split(key)
params = init_params(this_key)
value_and_grad_fn = jax.value_and_grad(loss_fn)

Now we can run our training loop.
For the parameter loop, we use `jax.tree.map` to recursively update the dictionary of all the trainable parameters.

In [60]:
learning_rate = 0.01
num_steps = 1000

print("\nStarting training...")
for step in range(num_steps):
    current_loss, grads = value_and_grad_fn(params, X, y)

    params = jax.tree.map(lambda p, g: p - learning_rate * g, params, grads)

    if step % 100 == 0:
        print(f"Step {step}, Loss: {current_loss:.4f}")

print("\nTraining complete!")
print("Learned parameters:")
print(f"Weights (W): {params['W']}")
print(f"Biases (b): {params['b']}")
print("(Expected values:")
print(f"Weights (W): {W}")
print(f"Biases (b): {b})")


Starting training...
Step 0, Loss: 75.9620
Step 100, Loss: 11.6211
Step 200, Loss: 1.8050
Step 300, Loss: 0.2839
Step 400, Loss: 0.0452
Step 500, Loss: 0.0073
Step 600, Loss: 0.0012
Step 700, Loss: 0.0002
Step 800, Loss: 0.0000
Step 900, Loss: 0.0000

Training complete!
Learned parameters:
Weights (W): [[2.99953475 3.99936659 4.99977022]
 [0.99957151 6.9992473  5.29962709]]
Biases (b): [5.19984097 1.89937567]
(Expected values:
Weights (W): [[3.  4.  5. ]
 [1.  7.  5.3]]
Biases (b): [5.2 1.9])


## Extra topics

If we have time, we can work through some of these together

### More control flow - loops the `JAX` way

### Custom derivative rules

### Linear regression with equinox